In [0]:
# Project         : Procurement Analytics using Databricks & Power BI
# Layer           : Gold
# Notebook        : Gold_Fact_Invoices
# Source          : invoices.csv
# Target          : procurement.gold.gold_fact_invoices

#
# Author          : V R Mutyala
# Created Date    : 21-Jul-2026
# Last Modified   : 21-Jul-2026
#
# Description
# -----------
# This notebook created the gold_fact_invoices table in the gold layer.
# This notebook creates the fact_invoices table in the gold layer.
# Includes invoice-vs-PO variance calculation, match status flagging,
# overdue/aging bucketing, and multi-currency conversion to USD.
#
# ==============================================================================
# Business Objective
# ==============================================================================
# Create the invoices Fact table from the Silver layer.
# ==============================================================================

In [0]:
%run ../01_Config/Config

In [0]:
%run ../05_Helper_Functions/Helper_functions

In [0]:
# ============================================================
# Read Silver invoices Table
# ============================================================

silver_invoices_df = read_delta(SILVER_INVOICES)

preview(silver_invoices_df,"Silver Invoices")

In [0]:

# Create Invoices Fact table with variance, FX & Match Flags
pyspark.sql.functions import (
    col,
    when,
    datediff,
    current_timestamp,
    current_date,
    round
)

# Load FX Rates Dimension

fx_rates_df = read_delta(GOLD_DIM_FX_RATES)

# Select Required Invoice Columns
gold_fact_invoices_df = (
    silver_invoices_df
        .select(
            "invoice_id",
            "invoice_date",
            "po_id",
            "po_amount_expected",
            "invoice_amount",
            "currency",
            "due_date",
            "invoice_status"
        )
)

# Currency Conversion + Variance + Match Status

gold_fact_invoices_df = (
    gold_fact_invoices_df

        # Join FX Rates
        .join(
            fx_rates_df,
            on="currency",
            how="left"
        )

# Convert original amounts to USD
       .withColumn(
            "invoice_amount_usd",
            (
                col("invoice_amount").cast("decimal(18,2)")
                * col("to_usd_rate").cast("decimal(18,6)")
            ).cast("decimal(18,2)")
        )

        .withColumn(
            "po_amount_expected_usd",
            (
                col("po_amount_expected").cast("decimal(18,2)")
                * col("to_usd_rate").cast("decimal(18,6)")
            ).cast("decimal(18,2)")
        )

# Original Currency Variance
        .withColumn(
            "invoice_variance_amount",
            (
                col("invoice_amount").cast("decimal(18,2)")
                - col("po_amount_expected").cast("decimal(18,2)")
            ).cast("decimal(18,2)")
        )

# USD Variance
        .withColumn(
            "invoice_variance_amount_usd",
            (
                col("invoice_amount_usd")
                - col("po_amount_expected_usd")
            ).cast("decimal(18,2)")
        )

# Variance Percentage
        .withColumn(
            "invoice_variance_pct",
            when(
                col("po_amount_expected") != 0,
                (
                    (
                        col("invoice_amount").cast("decimal(18,2)")
                        - col("po_amount_expected").cast("decimal(18,2)")
                    )
                    / col("po_amount_expected").cast("decimal(18,2)")
                    * 100
                ).cast("decimal(10,2)")
            ).otherwise(None)
        )

       
# Invoice Match Status
     
        .withColumn(
            "invoice_match_status",
            when(
                col("invoice_amount").cast("decimal(18,2)")
                == col("po_amount_expected").cast("decimal(18,2)"),
                "Matched"
            )
            .when(
                col("invoice_amount").cast("decimal(18,2)")
                > col("po_amount_expected").cast("decimal(18,2)"),
                "Overbilled"
            )
            .otherwise("Underbilled")
        )

# Remove FX rate after calculations
        .drop("to_usd_rate")
)

# Reporting / As-of Date

AS_OF_DATE = "2025-12-31"


# Overdue / Aging Calculations

gold_fact_invoices_df = (
    gold_fact_invoices_df

        
# Days overdue reporting date 
        .withColumn(
        "days_overdue",
        when(
            col("due_date") < lit(AS_OF_DATE),
            datediff(
                lit(AS_OF_DATE),
                col("due_date")
            )
        ).otherwise(0)
    )

        
# Overdue Flag
      
         .withColumn(
        "is_overdue",
        when(
            col("due_date") < lit(AS_OF_DATE),
            True
        ).otherwise(False)
    )

# Aging Bucket
      
       .withColumn(
        "aging_bucket",
        when(
            col("days_overdue") <= 0,
            "Current"
        )
        .when(
            col("days_overdue") <= 30,
            "1-30 Days"
        )
        .when(
            col("days_overdue") <= 60,
            "31-60 Days"
        )
        .when(
            col("days_overdue") <= 90,
            "61-90 Days"
        )
        .otherwise(
            "90+ Days"
        )
    )

      
# Gold Load Timestamp
     
        .withColumn(
            "gold_load_timestamp",
            current_timestamp()
        )
)


# Preview Gold Data

gold_fact_invoices_df.display()

In [0]:
# ============================================================
# Add Gold Metadata
# ============================================================

gold_fact_invoices_df = (
    gold_fact_invoices_df
        .withColumn(
            "gold_load_timestamp",
            current_timestamp()
        )
)

In [0]:
# ============================================================
# Write Gold Delta Table
# ============================================================

write_delta(
    df=gold_fact_invoices_df,
    table_name=GOLD_FACT_INVOICES
)

In [0]:
# ============================================================
# Validate Gold Delta Table
# ============================================================

validate_table(GOLD_FACT_INVOICES)

In [0]:
# ============================================================
# Gold Layer Summary
# ============================================================

print("=" * 60)
print("Gold Invoices Load Completed Successfully")
print("=" * 60)

print(f"Silver Records : {silver_invoices_df.count():,}")
print(f"Gold Records   : {gold_fact_invoices_df.count():,}")